# Mini-Project: Bespoke Retrieval-Augmented Generation (RAG)


## Overview


In this project, you will build a hybrid system that combines three components into a single RAG-style pipeline:

- A **language model (GPT-2)** for text generation.
- A **Query-Key Memory Network (QK-MemNet)** for factual retrieval.
- A **router** that decides whether to use autoregressive generation or to invoke the retrieval model.

This assignment combines language modeling, retrieval, text classification, and data curation.

The overall goal is to build a system that generates biographies of real people. At each step of generation, a router model examines the current text that has been generated so far and decides: "Should we **generate** the next token with GPT-2, or **retrieve** a fact from structured memory?"

**Note:** This system is **not expected to produce perfect biographies**. GPT-2 is a small model and the QK-MemNet uses bag-of-words representations. The goal is to demonstrate the *architecture*, decision-making between retrieval and generation, and integration of symbolic memory with neural models.

## Learning objectives

The learning objective of this assignment is to provide practical experience handling structured and unstructured data, training classifiers on real-world data, and incorporating different models into a single, practical generative system.

We ask you to envision that you have been tasked with creating a specialized system that generates biographies for famous people that tamps down on hallucinations as much as possible. With the very powerful LLMs available today, retrieval-augmented generation (RAG) might be achieved by using web search to pull some web pages for a person and then concatenating the contents of the web page along with a prompt, and letting a very large LLM do its thing with attention. However, this is not very exciting as an assignment that combines skills across the entire class.

Instead, we are going to pretend that we have an LLM that has not been able to pre-train on biographies of famous people (if you want, you can pretend that we are working on a system that generates biographies of un-famous people). To simulate this setting, we will work with GPT-2 because it is less capable of memorizing facts. This also means we can work with language models that are small enough to not need a lot of computing resources. No one will realistically use GPT-2, but GPT-2 has not memorized a lot of biographical information, so it suits our purposes.

We will also need a retriever. For this assignment, we will build a Query-Key Memory Network (QK-MemNet), inspired by Key-Value Memory Networks (KV-MemNets). A Memory Network learns to index into a data source (external memory) using attention mechanisms. They were considered a promising pathway to more sophisticated retrieval and generation capabilities before the Transformer came along. A Query-Key Memory Network assumes that memory is structured as a hash table and learns the mapping from an arbitrary query to the appropriate keys in order to retrieve its associated values. Where a hash table requires an exact match between query and key, a QK-MemNet learns a function that maps queries to keys. Memory networks use attention in a way that somewhat corresponds to Transformers. This makes sense because Transformers use their K, V, and Q sub-networks that are inspired by hash tables. We thus get to play with something that shares a lot of commonalities with Transformers without implementing all the complexity of the Transformer (and for which the code is already given earlier in the lectures).

The QK-MemNet will store biographical facts (birthdates, birth places, etc.), but not full biographies. So we will have GPT-2 generate biographies, and it will likely hallucinate the facts. How do we know when to retrieve? For this, we will train a classifier that will tell us whether the next token(s) should be retrieved or if we should let GPT-2 generate the next tokens. Thus the generation flow looks like this:

```
GPT-2 generates token → Classifier checks: is a fact needed next?
                            ↳ YES → QK-MemNet retrieves fact → inject into context
                            ↳ NO  → continue generating
```

The router will be a fine-tuned version of BERT, trained to predict whether the next token(s) should be facts. To fine-tune BERT, we will need to construct a training set of sentence fragments, some of which are followed by facts and some of which are not. Thus, we will need to prepare a specialized dataset. We will provide you with the raw materials to do this.

There are other ways this RAG project might have been done. We might have let the LM generate and then tried to detect hallucinated facts and "rewind" and invoke the retriever. That has its own challenges. We might fine-tune the LM to generate placeholder text and then grab those placeholders and call the retriever when a placeholder appears. We have chosen to insist on a particular structure for the assignment that involves creating datasets, training a classifier, and building a bespoke model architecture from scratch for their educational value.

By the end of this project, you will have practiced:

1. Using open-weight language models.
2. Creating supervised learning datasets that align structured knowledge with unstructured text.
3. Training a neural binary classifier off annotated data.
4. Implementing attention-based neural architectures for retrieval.
5. Building specialized RAG systems.

## How to use this Notebook


- Cells marked with `### START CODE HERE ###` and `### END CODE HERE ###` are where you write your solutions.
- Do **not** modify provided helper functions or test cells unless instructed.
- Run cells **in order**, later parts depend on earlier results.

The project is split across three notebooks:

- **Part A: Router**
- **Part B: Memory Network**
- **Parts C–D: GPT-2 fine-tuning (optional) and RAG integration**

This notebook covers Parts C and D. Use the router from Part A and the retriever from Part B.

## Grading Summary: Parts C–D

| Section | Title | Function(s) | Checks | Points |
| --- | --- | --- | --- | ---: |
| C | Prepare GPT-2 | — | — | 0 |
| D.1 | Employ the Router, Generator, and Retriever | `trigger_retrieval` | 1.1–1.2 | 1 |
| D.1 | Employ the Router, Generator, and Retriever | `generate` | 2.1–2.3 | 1.25 |
| D.1 | Employ the Router, Generator, and Retriever | `retrieve` | 3.1–3.3 | 1.25 |
| D.1 | Employ the Router, Generator, and Retriever | `generate_biography` | 4.1–4.5 | 3.5 |
| **Total** | | | | **7** |

The checks listed above are the tests used by the autograder. You can try them using the Student Self-check at the end of this notebook.


# Setting Up

In [ ]:
!pip install unidecode --quiet

In [ ]:
import json
import re
import random
from pathlib import Path
from typing import Any, Dict, List, Tuple, Optional
from copy import deepcopy
import unidecode

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

In [ ]:
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

The following cell mounts Google Drive to the Colab environment so that files stored in Drive can be accessed directly from the notebook. This is only necessary if you are using Google Colab.

After mounting, we define the root directory of the project located in Google Drive. This provides a centralized path for all file references in the project.

In [ ]:
PROJECT_ROOT = Path("/content/drive/MyDrive/cs7650_final_project")  # adjust as needed

In [ ]:
MOUNT_GOOGLE_DRIVE = True

if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception as e:
        print("Google Drive mount failed. If not on Google Colab, you can safely ignore this error and set MOUNT_GOOGLE_DRIVE to False.")
        print(e)

In [ ]:
# setup
# Load person_to_kv_all
def read_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))

ANNOTATION_OUTPUT_DIR = PROJECT_ROOT / "annotation"
PERSON_KV_ALL_PATH = ANNOTATION_OUTPUT_DIR / "person_to_kv_all.json"
person_to_kv_all = read_json(PERSON_KV_ALL_PATH)


# Section C: Prepare GPT-2 (0 points)

This prepares GPT-2 for autoregressive generation. If necessary, you can fine-tune GPT-2 on the Simple Wiki data to make it generate biographies that are more in the form of the data the router is also trained on. However, GPT-2 does not have a lot of capacity for memorization so the router and retriever will still be necessary.

## C.1. Load GPT-2 (0 points)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import Dataset
from transformers import DataCollatorForLanguageModeling

In [ ]:
gpt_tokenizer = AutoTokenizer.from_pretrained('gpt2-large')
# Set the pad token to the EOS token for GPT-2
gpt_tokenizer.pad_token = gpt_tokenizer.eos_token
gpt_model = AutoModelForCausalLM.from_pretrained('gpt2-large').to(DEVICE)
print(f"GPT-2 loaded on {DEVICE}")

## C.2. Fine-tune GPT-2 (optional, 0 points)


In [ ]:
trainer = None # Set this variable in the next block if fine-tuning

1. Create a Dataset object in the form of `{'text': list_of_biography_paragraphs}`
2. Write a pre-processing function that adds tokenizes each paragraph and adds the `tokenizer.eos_token` to the end and truncates (if necessary). Apply the pre-processing function as follows:
```
tokenized = raw_ds.map(
        my_preprocess_fn,
        batched=True,
        remove_columns=["text"],
        desc="Tokenizing paragraphs",
    )
```
3. Split the data into blocks by writing another pre-processing function that breaks into equal-sized blocks. Apply with the following:
```
dataset = tokenized.map(
        group_texts,
        batched=True,
        desc=f"Packing into {block_size} token blocks",
    )
```
4. Split the dataset into train and eval sets.
5. Create a Data collator: `DataCollatorForLanguageModeling(tokenizer=gpt_tokenizer, mlm=False)`
6. Create `TrainingArguments`
7. Create `Trainer` with the `gpt_model`, your training data split, eval data split, and data collator.

In [ ]:
### START CODE HERE ###





### END CODE HERE ###

In [ ]:
FINE_TUNE_GPT = False
USE_ALT_MODEL = False # Use the instructor-fine-tuned model

if FINE_TUNE_GPT:
  trainer.train()
  gpt_model = trainer.model
elif USE_ALT_MODEL:
  model_id = "specialsaucem/simple_wiki_bio_generator"
  gpt_model = AutoModelForCausalLM.from_pretrained(model_id).to(DEVICE)

Test the generator. You should see a biography that looks somewhat like the simple biographies.

In [ ]:
prompt = "Mark Riedl"
input_ids = gpt_tokenizer.encode(prompt, return_tensors="pt").to(DEVICE)

output = gpt_model.generate(
    input_ids,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.1,
    top_p=0.95,
    repetition_penalty=1.2,
    no_repeat_ngram_size=3,
    pad_token_id=gpt_tokenizer.eos_token_id,
)
print(gpt_tokenizer.decode(output[0], skip_special_tokens=True))

# Section D: Integrate router, retriever, and generator (7 points)

Combine the router from Part A, the retriever from Part B, and GPT-2 to generate biographies.


## D.0. Helper Data Structures

The following will allow our system to print in color. Tokens provided in a prompt will be cyan, tokens generated by autoregression will be blue, and tokens from retrieval will be red.

There are two data structures you need to know about here.

A `TokenSpan` object contains tokens in a tensor (shape 1xn). It also knows which tokenizer to use and how the tokens were created. Once a TokenSpan is created, it can be used like a string, e.g. `print()` and `str()` will produce colored text. Additionally, you can use `.tokens()` to get the raw tokens and `.raw_str()` to get uncolored text.

```
span = TokenSpan(torch.tensor([[9771, 7114, 15226, 3130, 373]]), gpt_tokenizer, SpanType.PROMPT)
print(span)
```
<font color="cyan" face="monospace" size="3">Calvin Coolidge was</font>

A `TokenDoc` object consists of a list of TokenSpans and also behaves like a string with `print()` and `str()`. TokenSpans, TokenDocs, token tensors, and tokens (ints) can be concatenated with the `+` operator.

```
doc = TokenDoc(default_tokenizer = gpt_tokenizer)
doc = doc + span
doc = doc + TokenSpan(torch.tensor([[6286, 319]]), gpt_tokenizer, SpanType.GENERATED)
doc = doc + TokenSpan(torch.tensor([[20, 3269, 26539]]), gpt_tokenizer, SpanType.RETRIEVED)
print(doc.tokens())
[[9771, 7114, 15226, 3130, 373, 6286, 319, 20, 3269, 26539]]
print(len(doc))
10
print(doc)
```
<font color="cyan" face="monospace" size="3">Calvin Coolidge was</font> <font color="blue" face="monospace" size="3">born on</font> <font color="red" face="monospace" size="3">5 January 1933</font>

It is likely that all tokenizers will be `gpt_tokenizer`.

In [ ]:
#!pip install termcolor

In [ ]:
from enum import Enum
from termcolor import colored
from functools import reduce

### there are 5 different types of token spans
class SpanType(Enum):
  GENERATED = 'blue'
  RETRIEVED = 'red'
  PROMPT = 'cyan'
  NONE = 'white'
  MANUAL = 'magenta'

### TokenSpan class
class TokenSpan:
  def __init__(self, tokens: torch.tensor,
               tokenizer: AutoTokenizer,
               span_type: SpanType = SpanType.NONE):
    self.tokens = tokens
    self.tokenizer = tokenizer
    self.span_type = span_type

  ### Make it so len() works with TokenSpans
  def __len__(self):
    return len(self.tokens)

  ### Make it so print() and str() work
  def __str__(self):
    return colored(self.decoded(), self.span_type.value, force_color=True)

  ### Apply the tokenizer to decode (returns uncolored text)
  def decoded(self):
    return self.tokenizer.decode(self.tokens)[0]

  ### Get the tokens
  def tokens(self):
    return self.tokens

  ### make deep copy work
  def __copy__(self):
    return TokenSpan(self.tokens[:], self.tokenizer, self.span_type)

### TokenDoc class
class TokenDoc:
  def __init__(self, default_tokenizer: AutoTokenizer,
               init_tokens: torch.tensor = None,
               default_span_type: SpanType = SpanType.NONE):
    self.spans = []
    self.default_tokenizer = default_tokenizer
    self.default_span_type = default_span_type
    if init_tokens is not None:
      self.spans.append(TokenSpan(init_tokens,
                                  tokenizer=self.default_tokenizer,
                                  span_type=self.default_span_type))

  ### Make print() and str() work
  def __str__(self):
    return ''.join(str(s) for s in self.spans)

  ### Get raw, uncolored text
  def raw_str(self):
    return ''.join([span.decoded() for span in self.spans])

  ### Make deep copy work
  def __copy__(self):
    doc = SpanDoc(default_tokenizer=self.tokenizer,
                  default_span_type=self.default_span_type)
    doc.spans = self.spans[:]
    return doc

  ### Add something to the SpanDoc
  def __add__(self, other):
    # Make a deep copy
    tokenizers = [self.default_tokenizer] + [span.tokenizer for span in self.spans]
    doc = deepcopy(self, {id(t): t for t in tokenizers})  # tokenizers are slow to copy and never change
    # Adding another SpanDoc
    if isinstance(other, TokenDoc):
      doc.spans = doc.spans + other.spans
    # Adding a TokenSpan
    elif isinstance(other, TokenSpan):
      if len(doc.spans) > 0 and other.span_type == doc.spans[-1].span_type and other.tokenizer.name_or_path == doc.spans[-1].tokenizer.name_or_path:
        # TokenSpan should be merged with last span
        doc.spans[-1].tokens = torch.cat([doc.spans[-1].tokens, other.tokens], dim=1)
      else:
        # Cannot be merged, add to list of spans
        doc.spans.append(other)
    # Adding an int
    elif isinstance(other, int):
      if len(doc.spans) == 0:
        doc.spans.append(TokenSpan(torch.tensor([[other]]),
                                   self.default_tokenizer,
                                   self.default_span_type))
      else:
        doc.spans[-1].tokens = torch.cat([doc.spans[-1].tokens, torch.tensor([[other]])], dim=1)
    # Adding a Tensor (of the correct type and shape)
    elif isinstance(other, torch.Tensor) and other.type() == torch.LongTensor and len(other.size()) == 2 and other.size()[0] == 1:
      if len(doc.spans) == 0:
        doc.spans.append(TokenSpan(other,
                                   self.default_tokenizer,
                                   self.default_span_type))
      else:
        doc.spans[-1].tokens = torch.cat([doc.spans[-1].tokens, other], dim=1)
    # Don't know how to merge
    else:
      raise ValueError("Cannot add this type of object to a SpanDoc.")
    return doc

  ### Make len() work (returns number of tokens)
  def __len__(self):
    return self.num_tokens()

  ### Get a span using indexing brackets
  def __getitem__(self, key):
    return self.spans[key]

  ### Return number of tokens
  def num_tokens(self):
    return self.tokens().size()[1]

  ### Return number of spans
  def num_spans(self):
    return len(self.spans)

  ### Get all the tokens as one big tensor
  def tokens(self):
    if len(self.spans) == 0:
      return torch.tensor([[]]).to(DEVICE)
    elif len(self.spans) == 1:
      return self.spans[0].tokens
    else:
      return torch.cat([span.tokens for span in self.spans], dim=1)

  ### Get all spans
  def spans(self):
    return self.spans

## D.1. Employ the Router, Generator, and Retriever (7 points)

Implement the `trigger_retrieval()` function, which takes a text (prefix), the router model and router tokenizer and returns `True` if the text should be followed by a retrieved fact or `False` otherwise.

In [ ]:
def trigger_retrieval(text, router_model, router_tokenizer):
  should_retrieve = False # Set this to true if a fact needs to be retrieved
  ### START CODE HERE ###





  ### END CODE HERE ###
  return should_retrieve

Implement `generate()` which produces a single token from GPT-2 (or your fine-tuned version).

The function will take the following:
- `model`: the GPT model, the input ids as a tensor of size 1xn, an attention mask, and position_ids.
- `input_ids`: tokens, in a tensor shape 1xn
- `attention_mask`: which positions to include in attention, tensor of length n.
- `position_ids`: which positions hvae new: initially a tensor containing `[0, 1, ..., n]`. When tokens are generated, the position_ids should receive the index of the latest token. When retrieval produces more than one token, make sure to add position_ids for all new tokens retrieved.
- `past_key_values`: this is the kv_cache used for efficiency.
- `tokenizer`: the tokenizer

The function should return a `TokenSpan` of type `SpanType.GENERATED`. Do not change the span type.

In [ ]:
def generate(model, input_ids, attention_mask = None, position_ids = None, past_key_values=None, tokenizer=None):
  next_id = None # The new token in a tensor (e.g.,[317])
  new_past_key_values = None # The new kv_cache
  ### START CODE HERE ###





  ### END CODE HERE ###
  return TokenSpan(next_id.unsqueeze(0), tokenizer, SpanType.GENERATED), new_past_key_values

Use the query to retrieve a fact from the QK-MemNet.

The function should return a `TokenSpan` of type `SpanType.RETRIEVED`. Do not change the span type.

In [ ]:
def retrieve(model, name, person_to_kv, query, vocab, tokenizer):
  retrieved_text = None # The text that is retrieved
  ### START CODE HERE ###





  ### END CODE HERE ###
  return TokenSpan(tokenizer.encode(retrieved_text, return_tensors="pt").to(DEVICE),
                   tokenizer, SpanType.RETRIEVED) if retrieved_text is not None else None

Implement `generate_biography()`. Parameters:
- `name`: the name of the individual to generate a biography for
- `prompt_bank`: a list of prompts. Prompts can have `{name}` in them, in which case the name of the individual will be substituted in. You need to iterate through all prompts in the prompt bank.
- `person_to_kv`: a dictionary mapping names to a list of (key, value) pairs.
- `gpt_model`: the GPT-2 or fine-tuned model.
- `gpt_tokenizer`: the tokenizer for gpt_model.
- `router_model`: your router model.
- `router_tokenizer`: your router model's tokenizer.
- `qkmem_model`: your retriever model.
- `vocab`: the Vocab object for your retriever.

For example, if the `prompt_bank` contains:
```
['{name} (', 'He was married']
```
then the output might be expected to be:

<font color="cyan" face="monospace" size="3">calvin coolidge (</font> <font color="red" face="monospace" size="3">4 July 1872</font> <font color="blue" face="monospace" size="3">–</font> <font color="red" face="monospace" size="3">5 January 1933</font><font color="blue" face="monospace" size="3">) was an American</font> <font color="red" face="monospace" size="3">politician.</font> <font color="cyan" face="monospace" size="3">He was married</font> <font color="blue" face="monospace" size="3">to</font> <font color="red" face="monospace" size="3">Grace Coolidge</font><font color="blue" face="monospace" size="3">.</font>

Where:
- <font color="cyan" face="monospace" size="3">cyan</font> indicates text from the `prompt_bank`
- <font color="blue" face="monospace" size="3">blue</font> indicates text generated from GPT-2
- <font color="red" face="monospace" size="3">red</font> indicates text retrieved from the DB.

(Note that in this example the first prompt in the `prompt_bank` generated three distinct retrievals).

The logic for this function is as follows:
1. iterate through each prompt in the `prompt_bank`.
2. Add the prompt to the document:
`document += TokenSpan(prompt_tokens, gpt_tokenizer, SpanType.PROMPT)`
3. Until the sentence is complete or until a max is reached...
4. Use the router to determine if the next token comes from GPT-2 or from the retriever.
5. Use the `generate()` or `retrieve()` function as appropriate. These functions will return a `TokenSpan`
6. Concatenate the returned `TokenSpan` to `document`.
7. Return the document, which will be a paragraph of at least `len(prompt_bank)` sentences.

You may write custom code to produce <font color="magenta" face="monospace" size="3">manual</font> text if you need to add punctuation, edit generated text, or add any other non-autoregressive and non-retrieved text. To do this you can concatenate a `TokenSpan(my_tokenized_text, gpt_tokenizer, SpanType.MANUAL)` to `document`.


In [ ]:
def generate_biography(name, prompt_bank, person_to_kv,
                       gpt_model, gpt_tokenizer,
                       router_model, router_tokenizer,
                       qkmem_model, vocab):
  # Make an empty TokenDoc
  document = TokenDoc(default_tokenizer = gpt_tokenizer, default_span_type = SpanType.NONE)
  ### START CODE HERE ###





  ### END CODE HERE ###
  return document

In [ ]:
import spacy
nlp = spacy.blank("en")

def tokenize(text: str) -> List[str]:
    doc = nlp.make_doc(text.replace('_', ' '))
    return [unidecode.unidecode(tok.text.lower()) for tok in doc]

class Vocab:
    def __init__(self):
        self._word2index = {}
        self._index2word = {}
        self._n_words = 0

    def num_words(self) -> int:
        return self._n_words

    def word2index(self, word: str) -> int:
        return self._word2index.get(word, -1)

    def index2word(self, idx: int) -> str:
        return self._index2word[idx]

    def add_word(self, word: str):
        if word not in self._word2index:
            self._word2index[word] = self._n_words
            self._index2word[self._n_words] = word
            self._n_words += 1

    def add_text(self, text: str):
        for token in tokenize(text):
            self.add_word(token)

    def __contains__(self, word: str) -> bool:
        return word in self._word2index

    def to_dict(self) -> dict:
        return dict(self._word2index)

    @classmethod
    def from_dict(cls, word2index: dict) -> "Vocab":
        vocab = cls()
        for word, idx in sorted(word2index.items(), key=lambda kv: kv[1]):
            vocab._word2index[word] = idx
            vocab._index2word[idx] = word
        vocab._n_words = len(vocab._word2index)
        return vocab

Paste your `QKMemNet` class from Part B into the cell below. `torch.load` in the next cell rebuilds your saved retriever, and it can only do that if the class it was saved from is defined here.

In [ ]:
### Paste your QKMemNet class from Part B here, so torch.load can rebuild your saved retriever.
class QKMemNet(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int):
        super(QKMemNet, self).__init__()
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        ### START CODE HERE ###

        ### END CODE HERE ###

    def forward(self, x: Any, keys: Any) -> torch.Tensor:
        ### START CODE HERE ###

        ### END CODE HERE ###
        return p

Run the pipeline with your trained router and retriever.

In [ ]:
from transformers import AutoModelForSequenceClassification

name = 'calvin coolidge'
prompt_bank = ['{name} (', 'He was married']
CLASSIFIER_OUTPUT_DIR = Path(PROJECT_ROOT) / "models" / f"router"
MEMNET_OUTPUT_DIR = Path(PROJECT_ROOT) / "models" / f"memnet"
vocab_file_path = PROJECT_ROOT / "vocab.json"

router_model = AutoModelForSequenceClassification.from_pretrained(str(CLASSIFIER_OUTPUT_DIR)).to(DEVICE)
qkmem_model = torch.load(MEMNET_OUTPUT_DIR / "qkmemnet.pt", weights_only=False)
VOCAB = Vocab.from_dict(read_json(vocab_file_path))
router_tokenizer = AutoTokenizer.from_pretrained(str(CLASSIFIER_OUTPUT_DIR))
doc = generate_biography(name, prompt_bank, person_to_kv_all, gpt_model, gpt_tokenizer, router_model, router_tokenizer, qkmem_model, VOCAB)
print(doc)

In [ ]:
def count_phrases(phrases, sentence, ignore_case=True, whole_words=False, used=set()):
    flags = re.IGNORECASE if ignore_case else 0
    spans = []
    for p in phrases:
        pattern = re.escape(p)
        if whole_words:
            pattern = rf"\b{pattern}\b"
        for m in re.finditer(pattern, sentence, flags):
            spans.append((m.end(), m.start(), p))   # (end, start, phrase)

    spans.sort()                  # earliest-ending first
    count = 0
    last_end = -1
    for end, start, phrase in spans:
        if start >= last_end and phrase not in used:
            count += 1
            used.add(phrase)
            last_end = end
    return count, used

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")
print("spaCy loaded.")

def score_biography(name, text, person_to_kv, prompt_bank):
  doc = nlp(text)
  used = set()
  for sent in doc.sents:
    print(sent.text)
    count, used = count_phrases([v for k,v in person_to_kv[name]], sent.text,
                                used=used)
  return len(used) / len(prompt_bank)

In [ ]:
score_biography(name, str(doc), person_to_kv_all, prompt_bank)

In [ ]:
person_to_kv_all['calvin coolidge']

In [ ]:
len(list(person_to_kv_all.keys()))

### Student Self-check/local autograder:

Put `test_part_C.py` in your working directory and run the cell below. These checks match the autograder.

In [ ]:
from test_part_C import run_all
run_all(trigger_retrieval, generate, retrieve, generate_biography, person_to_kv_all, gpt_model, gpt_tokenizer, router_model, router_tokenizer, qkmem_model, VOCAB)


Submit your completed `.ipynb` file to **HW5 Part C D** on Gradescope.
